<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 04. Dropout — Dropout: A Simple Way to Prevent Neural Networks from Overfitting

- **원 논문:** [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf)
- **저자 / 발표:** Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov · Journal of Machine Learning Research 15 (2014)
- **난이도 / 예상 시간:** 초중급 · 약 75분
- **선수 지식:** MLP, Bernoulli sampling, train/eval mode, expectation

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

논문 §2의 retain probability p를 사용하는 Bernoulli mask와 test-time weight scaling을 직접 구현한다. train/test 출력의 기대값 일치, masking 비율, 모듈 모드 전환을 검증하고 작은 XOR-like 분류기에서 loss 및 decision boundary를 관찰한다.

**원 논문과 다른 것**

원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, Fig. 1–2: 각 hidden unit을 probability p로 retain | `paper_dropout`의 Bernoulli mask | 10,000개 unit의 실제 retain fraction이 p 근처인지 assert |
| §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y | training branch의 `mask * x` | 고정 generator로 mask와 출력의 재현성 확인 |
| §2 test time: outgoing weights에 p를 곱한 scaled network | activation 관점에서 같은 `p * x`를 반환하는 eval branch | train Monte-Carlo mean과 deterministic eval mean 비교 |
| §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰 | `DropoutMLP` hidden activation mask와 합성 분류 실험 | loss curve와 2D decision probability map을 표시 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
m_i\sim\operatorname{Bernoulli}(q),\qquad
h_i^{\mathrm{train}}=m_i h_i,\qquad
h_i^{\mathrm{test}}=q h_i
$$

- $q=1-p$는 unit 보존 확률이고 $m_i$는 학습 시 샘플되는 mask입니다.
- 위 식은 원 논문의 관례입니다. train에서는 mask만 곱하고 test에서는 outgoing weight를
  $q$배하며, 실습은 동치인 activation $q h_i$를 계산합니다.
- PyTorch의 inverted dropout은 train에서 $m_i h_i/q$, eval에서 $h_i$를 사용합니다.
  두 관례의 기대값은 같지만 각 mode의 중간 activation scale은 다릅니다.
- 실습은 원 논문 관례의 mask, mode 전환, PyTorch 관례와의 차이를 함수로 확인합니다.
- activation shape은 유지되지만 train mode에서만 원소별 0이 생깁니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain
- **노트북 구현:** `paper_dropout`의 Bernoulli mask
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. 논문식 dropout과 PyTorch식 dropout

논문은 training 때 `mask*x`를 쓰고 test 때 weights를 `p`배 합니다. 이 실습은 activation
관점에서 test 출력을 `p*x`로 계산합니다. PyTorch의 `nn.Dropout(dropout_probability)`은
**inverted dropout**이라 training 때 살아남은 값을 `1/p`배하고 eval에서는 그대로 둡니다.
두 관례의 기대값은 같지만 중간 tensor의 scale이 다릅니다. 여기서 `p`는 drop 확률이
아니라 **retain 확률**임에 주의하세요.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,torch.tensor,torch.randn,torch.cat,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(4)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_xor_blobs(points_per_blob=20):
    generator = torch.Generator().manual_seed(505)
    centers = torch.tensor([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])
    labels = torch.tensor([0, 1, 1, 0])
    points, targets = [], []
    for center, label in zip(centers, labels):
        noise = 0.22 * torch.randn(points_per_blob, 2, generator=generator)
        points.append(center + noise)
        targets.append(label.repeat(points_per_blob))
    return torch.cat(points), torch.cat(targets)


drop_x, drop_y = make_xor_blobs()
drop_x, drop_y = ACCELERATOR.move(drop_x, drop_y)
assert drop_x.shape == (80, 2)
assert drop_y.bincount().tolist() == [40, 40]
print(ACCELERATOR.summary())
print("XOR-like batch:", tuple(drop_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain / §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y / §2 test time: outgoing weights에 p를 곱한 scaled network
- **이 셀의 책임:** `paper_dropout`의 Bernoulli mask / training branch의 `mask * x` / activation 관점에서 같은 `p * x`를 반환하는 eval branch
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert / 고정 generator로 mask와 출력의 재현성 확인 / train Monte-Carlo mean과 deterministic eval mean 비교. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.ones,Tensor.mean,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def paper_dropout(x, retain_probability=0.5, training=True, generator=None):
    if not 0.0 < retain_probability <= 1.0:
        raise ValueError("retain_probability must be in (0, 1]")
    if not training:
        return retain_probability * x
    probabilities = torch.full_like(x, retain_probability)
    mask = torch.bernoulli(probabilities, generator=generator)
    return mask * x


ones = torch.ones(10_000)
dropout_generator = torch.Generator().manual_seed(606)
sampled = paper_dropout(ones, 0.6, True, dropout_generator)
evaluated = paper_dropout(ones, 0.6, False)
retain_fraction = sampled.mean().item()
assert abs(retain_fraction - 0.6) < 0.02
assert torch.allclose(evaluated, torch.full_like(ones, 0.6))
assert abs(sampled.mean().item() - evaluated.mean().item()) < 0.02
print(
    f"sampled retain={retain_fraction:.3f}; "
    f"deterministic test mean={evaluated.mean():.3f}"
)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Fig. 1–2: 각 hidden unit을 probability p로 retain / §2, Eq. (1): r_j^(l) ~ Bernoulli(p), y~=r*y / §2 test time: outgoing weights에 p를 곱한 scaled network / §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `paper_dropout`의 Bernoulli mask / training branch의 `mask * x` / activation 관점에서 같은 `p * x`를 반환하는 eval branch / `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 10,000개 unit의 실제 retain fraction이 p 근처인지 assert / 고정 generator로 mask와 출력의 재현성 확인 / train Monte-Carlo mean과 deterministic eval mean 비교 / loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,F.relu,Module.eval,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class PaperDropout(nn.Module):
    def __init__(self, retain_probability=0.5):
        super().__init__()
        if not 0.0 < retain_probability <= 1.0:
            raise ValueError("retain_probability must be in (0, 1]")
        self.retain_probability = retain_probability

    def forward(self, x):
        return paper_dropout(x, self.retain_probability, self.training)


class DropoutMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 24)
        self.fc2 = nn.Linear(24, 24)
        self.out = nn.Linear(24, 2)
        self.dropout = PaperDropout(retain_probability=0.75)

    def forward(self, x):
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.out(x)


torch.manual_seed(4)
dropout_model = DropoutMLP().to(DEVICE)
dropout_model.eval()
first_eval = dropout_model(drop_x[:4])
second_eval = dropout_model(drop_x[:4])
assert torch.equal(first_eval, second_eval)
assert first_eval.shape == (4, 2)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Module.train,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,Tensor.detach,torch.no_grad,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
dropout_optimizer = torch.optim.Adam(dropout_model.parameters(), lr=0.03)
dropout_losses = []
dropout_model.train()
for _ in range(60):
    dropout_optimizer.zero_grad()
    loss = F.cross_entropy(dropout_model(drop_x), drop_y)
    loss.backward()
    dropout_optimizer.step()
    dropout_losses.append(float(loss.detach()))

dropout_model.eval()
with torch.no_grad():
    dropout_accuracy = (dropout_model(drop_x).argmax(1) == drop_y).float().mean().item()
assert torch.isfinite(torch.tensor(dropout_losses)).all()
assert min(dropout_losses[1:]) < dropout_losses[0]
print(
    f"loss {dropout_losses[0]:.3f} → {dropout_losses[-1]:.3f}; "
    f"accuracy={dropout_accuracy:.1%}"
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §7.2 ‘Effect of Dropout on Sparsity’: sparse activations 관찰
- **이 셀의 책임:** `DropoutMLP` hidden activation mask와 합성 분류 실험
- **Tensor shape 계약:** activation [B,H] → 같은 shape의 masked/scaled activation [B,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss curve와 2D decision probability map을 표시. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.stack,Tensor.flatten -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>tensor.flatten(start_dim=0, end_dim=-1)</code></summary>

#### `tensor.flatten(start_dim=0, end_dim=-1)`

- **역할:** 지정한 연속 축들을 하나의 축으로 합칩니다.
- **입력·반환:** 축 범위를 받고 평탄화된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** feature map을 classifier 입력이나 토큰 열로 바꿉니다.
- **주의점:** batch 축까지 실수로 합치지 않도록 `start_dim`을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.flatten](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.flatten.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
grid_axis = torch.linspace(-1.7, 1.7, 80, device=DEVICE)
grid_y, grid_x = torch.meshgrid(grid_axis, grid_axis, indexing="ij")
grid_points = torch.stack((grid_x.flatten(), grid_y.flatten()), dim=1)
dropout_model.eval()
with torch.no_grad():
    class_one_probability = dropout_model(grid_points).softmax(1)[:, 1].reshape(80, 80)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
image = axes[0].contourf(
    grid_x.detach().cpu(),
    grid_y.detach().cpu(),
    class_one_probability.detach().cpu(),
    levels=20,
    cmap="coolwarm",
)
axes[0].scatter(
    drop_x[:, 0].detach().cpu(),
    drop_x[:, 1].detach().cpu(),
    c=drop_y.detach().cpu(),
    cmap="coolwarm",
    edgecolor="black",
    s=18,
)
axes[0].set_title("test-time scaled decision map")
fig.colorbar(image, ax=axes[0], fraction=0.046)
axes[1].plot(dropout_losses)
axes[1].set_title("stochastic training loss")
axes[1].set_xlabel("optimization step")
axes[1].set_ylabel("cross-entropy")
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()
print(
    "decision probability range:",
    float(class_one_probability.min()),
    float(class_one_probability.max()),
)

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 MNIST, TIMIT, CIFAR-10/100, SVHN, ImageNet에서 큰 networks와 여러 hyperparameter를 비교한다. 여기서는 80개의 2D Gaussian points와 2-layer MLP만 사용해 stochastic thinning과 test-time scaling 메커니즘을 검증한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.